CUSTOMER_SILVER


In [0]:
%sql

-- FIXED: Added ROW_NUMBER deduplication to ensure only 1 record per CustomerID
CREATE OR REPLACE TABLE silver_catalog.retail_silver.silver_customers AS
SELECT
    CustomerID,
    CustomerName,
    Email,
    City,
    Address,
    LastUpdated
FROM (
    SELECT
        CAST(CustomerID AS INT) AS CustomerID,
        INITCAP(TRIM(CustomerName)) AS CustomerName,
        LOWER(TRIM(Email)) AS Email,
        TRIM(City) AS City,
        TRIM(Address) AS Address,
        TO_DATE(LastUpdated,'dd-MM-yyyy') AS LastUpdated,
        ROW_NUMBER() OVER (
            PARTITION BY CustomerID 
            ORDER BY TO_DATE(LastUpdated,'dd-MM-yyyy') DESC, CustomerName DESC
        ) AS rn
    FROM bronze_catalog.retail_bronze.bronze_customers
    WHERE CustomerID IS NOT NULL
)
WHERE rn = 1;

In [0]:
%sql
-- Verify: Each CustomerID should appear exactly once
-- This query should return 0 rows if deduplication is working

SELECT 
    CustomerID,
    COUNT(*) AS RecordCount
FROM silver_catalog.retail_silver.silver_customers
GROUP BY CustomerID
HAVING COUNT(*) > 1;

PROUDCTS_SILVER


In [0]:
%sql
CREATE OR REPLACE TABLE silver_catalog.retail_silver.silver_products AS

SELECT DISTINCT
    CAST(ProductID AS INT) AS ProductID,
    TRIM(ProductName) AS ProductName,
    TRIM(Category) AS Category,
    CAST(UnitPrice AS DECIMAL(10,2)) AS UnitPrice
FROM bronze_catalog.retail_bronze.bronze_products
WHERE ProductID IS NOT NULL
AND UnitPrice > 0;

STORES_SILVER

In [0]:
%sql
CREATE OR REPLACE TABLE silver_catalog.retail_silver.silver_stores AS

SELECT DISTINCT
    CAST(StoreID AS INT) AS StoreID,
    INITCAP(TRIM(StoreName)) AS StoreName,
    TRIM(Region) AS Region
FROM bronze_catalog.retail_bronze.bronze_stores
WHERE Region IS NOT NULL;

SALES_SILVER

In [0]:
%sql
CREATE OR REPLACE TABLE silver_catalog.retail_silver.silver_sales AS
SELECT DISTINCT
    CAST(TransactionID AS INT) AS TransactionID,
    CAST(CustomerID AS INT) AS CustomerID,
    CAST(ProductID AS INT) AS ProductID,
    CAST(StoreID AS INT) AS StoreID,
    CAST(Quantity AS INT) AS Quantity,
    TO_DATE(TxnDate, 'dd-MM-yyyy') AS TxnDate
FROM bronze_catalog.retail_bronze.bronze_sales
WHERE TransactionID IS NOT NULL
AND Quantity > 0;

In [0]:
%sql
select  count(*) from silver_catalog.retail_silver.silver_customers UNION ALL
select  count(*) from silver_catalog.retail_silver.silver_products UNION ALL
select  count(*) from silver_catalog.retail_silver.silver_stores UNION ALL
select  count(*) from silver_catalog.retail_silver.silver_sales;


In [0]:
%sql
DROP TABLE IF EXISTS silver_catalog.retail_silver.silver_customers;
DROP TABLE IF EXISTS silver_catalog.retail_silver.silver_products;
DROP TABLE IF EXISTS silver_catalog.retail_silver.silver_sales;
DROP TABLE IF EXISTS silver_catalog.retail_silver.silver_stores;